# Slack 官方問答區 - 跨天池化精確縫合轉換工具

本 Notebook 實作**跨天訊息池化機制（Cross-Day Message Pooling & Thread Stitching）**：
1. **全域訊息池化**：讀取 `slackdump/官方問答區/json` 中所有的 JSON 檔案，以時間戳（`ts`）為 key 進行全域去重與集中池化。
2. **跨天線程精確縫合**：依 `thread_ts` 聚合每一個討論串，不受提問與回答跨天數、時區（美西 vs 台灣時差）或檔案分割影響，將提問、參賽者後續互動與官方多次回覆精確縫合。
3. **單一 Markdown 彙整輸出**：產出單一完整 Markdown 文件至 `slackdump/官方問答區/markdown/Slack_官方問答彙整_YYYYMMDD.md`，專供 NotebookLM 與 LLM 檢索。

In [2]:
from collections import defaultdict
from datetime import datetime
import glob
import json
import os
from pathlib import Path
import re

# ================= 參數與路徑設定 =================
OFFICIAL_UID = "U0C3LFZUG3D"  # 挑戰賽工作小組（官方）Slack User ID

# 自動判斷專案根目錄（支援從 code 目錄或專案根目錄執行）
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "slackdump").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "slackdump").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# 當天日期 (格式: YYYY-MM-DD)
today = datetime.now().strftime("%Y%m%d")

# 輸入 (JSON) 與 輸出 (Markdown) 資料夾與檔案路徑
INPUT_DIR = PROJECT_ROOT / "slackdump" / "官方問答區" / "json"
OUTPUT_DIR = PROJECT_ROOT / "slackdump" / "官方問答區" / "markdown"
OUTPUT_FILE = OUTPUT_DIR / f"Slack_官方問答彙整_{today}.md"

print(f"專案根目錄: {PROJECT_ROOT.resolve()}")
print(f"輸入目錄 (JSON):     {INPUT_DIR.resolve()}")
print(f"輸出目標 (Markdown): {OUTPUT_FILE.resolve()}")


專案根目錄: H:\我的雲端硬碟\DataScience\2026玉山人工智慧賽
輸入目錄 (JSON):     H:\我的雲端硬碟\DataScience\2026玉山人工智慧賽\slackdump\官方問答區\json
輸出目標 (Markdown): H:\我的雲端硬碟\DataScience\2026玉山人工智慧賽\slackdump\官方問答區\markdown\Slack_官方問答彙整_20261023.md


In [3]:
def clean_slack_text(text: str) -> str:
    """去除 Slack 內部超連結與使用者標籤，轉為標準 Markdown 格式"""
    if not text:
        return ""
    # 轉換 Slack 連結語法 <url|label> 為 [label](url)
    text = re.sub(r"<(https?://[^|>]+)\|([^>]+)>", r"[\2](\1)", text)
    # 轉換純 URL <url> 為 url
    text = re.sub(r"<(https?://[^>]+)>", r"\1", text)
    # 匿名化參賽者標籤 <@USER_ID> 為 @參賽者
    text = re.sub(r"<@[A-Z0-9]+>", "@參賽者", text)
    return text.strip()


def ts_to_datetime_str(ts_str: str) -> str:
    """將 Unix Timestamp 轉為 YYYY-MM-DD HH:MM:SS（本地時間）"""
    try:
        ts_float = float(ts_str)
        return datetime.fromtimestamp(ts_float).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return "未知時間"


In [ ]:
def convert_slack_to_unified_markdown(
    input_path: Path = INPUT_DIR,
    output_filepath: Path = OUTPUT_FILE,
):
    """
    跨天池化縫合機制：
    1. 掃描所有 JSON 檔案，將所有訊息載入全域訊息池 (all_messages)
    2. 依 thread_ts 進行線程聚合與時序排列（精確縫合跨多天發送的問答）
    3. 篩選出具有官方回覆 (OFFICIAL_UID) 或官方公告的討論串
    4. 依時間軸排序，整合輸出為單一 Markdown 文件供 NotebookLM 檢索
    """
    json_files = sorted(glob.glob(os.path.join(str(input_path), "*.json")))
    if not json_files:
        print(f"[Error] 找不到任何 .json 檔案，請確認路徑是否存在: {input_path}")
        return

    os.makedirs(output_filepath.parent, exist_ok=True)

    # 1. 全域訊息池化 (Global Message Pool)
    print(f"[*] 正在從 {len(json_files)} 個 JSON 檔案載入所有訊息池...")
    all_messages = {}
    for f in json_files:
        try:
            with open(f, "r", encoding="utf-8") as jf:
                data = json.load(jf)
                msgs = (
                    data if isinstance(data, list) else data.get("messages", [])
                )
                for m in msgs:
                    if m.get("type") == "message" and "ts" in m:
                        # 以唯一時間戳 ts 進行池化去重與更新
                        all_messages[m["ts"]] = m
        except Exception as e:
            print(f"[Warning] 讀取 {f} 失敗: {e}")

    print(f"[*] 訊息池共載入 {len(all_messages)} 則不重複訊息。")

    # 2. 跨天線程聚合與時序縫合 (Thread Stitching)
    threads = defaultdict(list)
    for ts, m in all_messages.items():
        th_id = m.get("thread_ts") or ts
        threads[th_id].append(m)

    # 針對每個討論串內部依發送時間排序
    for th_id in threads:
        threads[th_id].sort(key=lambda x: float(x["ts"]))

    # 3. 篩選官方參與的討論串 (Official Threads)
    official_threads = []
    total_official_replies = 0

    for th_id, msgs in threads.items():
        official_msgs = [m for m in msgs if m.get("user") == OFFICIAL_UID]
        if official_msgs:
            official_threads.append((th_id, msgs, official_msgs))
            total_official_replies += len(official_msgs)

    # 依整個議題起始時間（th_id timestamp）進行先後排序
    official_threads.sort(key=lambda x: float(x[0]))
    print(
        f"[*] 縫合完成：共過濾出 {len(official_threads)} 個官方議題討論串（含 {total_official_replies} 則官方回覆）。"
    )

    # 4. 寫入整合 Markdown 文件
    earliest_time = ts_to_datetime_str(official_threads[0][0])
    latest_time = ts_to_datetime_str(official_threads[-1][1][-1]["ts"])

    with open(output_filepath, "w", encoding="utf-8") as out:
        out.write("# 玉山 AI Agent 挑戰賽 - 官方問答完整彙整 (FAQ)\n\n")
        out.write(
            f"> 本文件由跨天訊息池化精確縫合產生，專供 NotebookLM 與大語言模型檢索。\n"
            f"> - 涵蓋議題數：共 {len(official_threads)} 則主題串\n"
            f"> - 官方答覆數：共 {total_official_replies} 則官方回覆\n"
            f"> - 時間範圍：{earliest_time} 至 {latest_time}\n"
            f"> - 官方身分：挑戰賽工作小組 (`{OFFICIAL_UID}`)\n\n"
            f"---\n\n"
        )

        for idx, (th_id, msgs, official_msgs) in enumerate(
            official_threads, 1
        ):
            root_msg = all_messages.get(th_id)

            if root_msg is not None:
                is_announcement = root_msg.get("user") == OFFICIAL_UID
                root_time = ts_to_datetime_str(root_msg["ts"])

                if is_announcement:
                    out.write(f"## 📢 議題 #{idx} 【主辦單位主動公告】\n\n")
                    out.write(f"- **發布時間**：`{root_time}`\n")
                    out.write(f"- **討論串 ID**：`{th_id}`\n\n")
                    out.write("### 📌 公告內容：\n")
                    out.write(
                        f"{clean_slack_text(root_msg.get('text', ''))}\n\n"
                    )
                else:
                    out.write(f"## 📌 議題 / 問題 #{idx}\n\n")
                    out.write(f"- **提問時間**：`{root_time}`\n")
                    out.write(f"- **討論串 ID**：`{th_id}`\n\n")
                    out.write("### 🙋‍♂️ 提問內容：\n")
                    out.write(
                        f"{clean_slack_text(root_msg.get('text', '（無提問內文）'))}\n\n"
                    )
                subsequent_msgs = [m for m in msgs if m["ts"] != th_id]
            else:
                # 根訊息發布日期早於本次 JSON 匯出起始日
                root_time = ts_to_datetime_str(th_id)
                out.write(f"## 📌 議題 / 問題 #{idx}\n\n")
                out.write(f"- **提問時間**：`{root_time}` (預估)\n")
                out.write(f"- **討論串 ID**：`{th_id}`\n\n")
                out.write("### 🙋‍♂️ 提問內容：\n")
                out.write(
                    f"*(原始提問時間為 {root_time}，因早於本次 Slack 匯出紀錄起始日故無提問原文，以下為官方答覆)*\n\n"
                )
                subsequent_msgs = msgs

            # 依時序輸出後續對話（含官方回覆與參賽者追問）
            reply_idx = 1
            for m in subsequent_msgs:
                msg_time = ts_to_datetime_str(m["ts"])
                msg_text = clean_slack_text(m.get("text", ""))

                if m.get("user") == OFFICIAL_UID:
                    label = (
                        f"（官方回覆 #{reply_idx}）"
                        if len(official_msgs) > 1
                        else ""
                    )
                    out.write(
                        f"### 💡 挑戰賽工作小組（官方回覆）{label}：\n"
                    )
                    out.write(f"*回覆時間: {msg_time}*\n\n")
                    out.write(f"{msg_text}\n\n")
                    reply_idx += 1
                else:
                    # 參賽者後續補充或追問
                    out.write(
                        f"> 💬 **參賽者回覆/補充** (*{msg_time}*)：\n"
                    )
                    out.write(f"> {msg_text}\n\n")

            out.write("---\n\n")

    print(
        f"[Success] 跨天池化縫合完成！已成功輸出至: {output_filepath}"
    )

In [ ]:
# 執行跨天池化縫合與單一 Markdown 彙整輸出
convert_slack_to_unified_markdown()

[*] 正在從 28 個 JSON 檔案載入所有訊息池...
[*] 訊息池共載入 136 則不重複訊息。
[*] 縫合完成：共過濾出 50 個官方議題討論串（含 54 則官方回覆）。
[Success] 跨天池化縫合完成！已成功輸出至: h:\我的雲端硬碟\DataScience\2026玉山人工智慧賽\slackdump\官方問答區\markdown\Slack_官方問答彙整_20261023.md
